In [42]:
# ------------------------------------------------------------
# 필요한 라이브러리 설치
# ------------------------------------------------------------
# !pip install transformers datasets evaluate sentencepiece rouge_score accelerate --upgrade

In [28]:

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # 토크나이저 멀티스레딩 경고 방지

from dataclasses import dataclass
from typing import Dict, List, Any

import numpy as np
from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)


In [29]:

# ------------------------------------------------------------
# 0) 모델 이름 지정 (KoBART)
# ------------------------------------------------------------
MODEL_NAME = "gogamza/kobart-base-v2"   # 한국어 BART 모델

# ------------------------------------------------------------
# 1) 예시 데이터 (입력 document + target summary)
#    → 학습 파이프라인 구조를 보여주기 위한 미니 샘플
# ------------------------------------------------------------
train_docs = [
    "정부는 중소기업 세제 혜택과 R&D 세액 공제를 확대한다고 밝혔다.",
    "해당 기업은 분기 실적에서 매출 성장을 기록했으며 신제품 출시를 예고했다.",
]
train_sums = [
    "정부가 중소기업 지원을 확대한다.",
    "기업이 실적 개선과 신제품 출시를 발표했다."
]

valid_docs = [
    "교육부는 내년부터 디지털 교과서 도입을 추진한다고 발표했다.",
]
valid_sums = [
    "교육부가 디지털 교과서 도입을 추진한다."
]

# HuggingFace datasets 형식으로 변환
raw_ds = DatasetDict({
    "train": Dataset.from_dict({"document": train_docs, "summary": train_sums}),
    "validation": Dataset.from_dict({"document": valid_docs, "summary": valid_sums}),
})
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 1
    })
})

In [30]:

# ------------------------------------------------------------
# 2) 토크나이저 / 모델 로드
#    KoBART는 SentencePiece 기반이므로 반드시 sentencepiece 설치 필요
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# 입력/출력 문장 최대 길이
max_input_len = 512
max_target_len = 128


You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.


In [ ]:

# ------------------------------------------------------------
# 3) 데이터 전처리 함수
#    Trainer는 tokenized 데이터셋을 요구 → map()으로 변환
# ------------------------------------------------------------
def preprocess_fn(batch):
    # ------------------------------
    # (1) 입력 문서 인코딩
    # ------------------------------
    inputs = tokenizer(
        batch["document"],
        max_length=max_input_len,
        padding="max_length",    # 고정 길이 패딩
        truncation=True,         # 길면 자르기
    )

    # ------------------------------
    # (2) 요약(정답) 인코딩
    #     → target tokenization
    #     → pad token은 -100으로 바꿔 loss 계산에서 제외
    # ------------------------------
    # 디코더용 토큰나이저를 사용
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["summary"],
            max_length=max_target_len,
            padding="max_length",
            truncation=True,
        )

    # pad 토큰을 -100으로 변경 (CrossEntropyLoss ignore_index)
    labels_ids = np.array(labels["input_ids"])
    labels_ids[labels_ids == tokenizer.pad_token_id] = -100
    inputs["labels"] = labels_ids.tolist()

    return inputs

# raw_ds에 전처리 적용
tokenized_ds = raw_ds.map(
    preprocess_fn,
    batched=True,
    remove_columns=["document", "summary"]   # 원본 텍스트는 삭제
)

# ------------------------------------------------------------
# 4) DataCollator
#    padding 동적 처리 + 모델 입력 형태 자동 구성
# ------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)


Map:   0%|          | 0/2 [00:00<?, ? examples/s]c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:3959: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  "You cannot use ``already_has_special_tokens=False`` with this tokenizer. "
Map: 100%|██████████| 1/1 [00:00<00:00, 322.34 examples/s]


In [43]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})

In [ ]:

# ------------------------------------------------------------
# 5) ROUGE Metric 설정
#    (요약 모델에서는 텍스트 생성 지표 필요)
# ------------------------------------------------------------
rouge = evaluate.load("rouge")

def postprocess_text(texts):
    # Rouge 계산을 위해 텍스트 양쪽 공백 정리
    return [t.strip() for t in texts]

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # -100(ignored index)을 다시 pad token으로 복원해야 디코딩 가능
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # 텍스트 디코딩
    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_str = postprocess_text(pred_str)
    label_str = postprocess_text(label_str)

    # ROUGE 계산
    result = rouge.compute(
        predictions=pred_str,
        references=label_str,
        use_stemmer=True
    )
    # use_stemmer : ROUGE 점수를 계산할 때 단어의 “원형(어간)” 기준으로 비교해라는 의미

    # % 단위로 보기 쉽게 소수점 정리
    result = {k: round(v * 100, 2) for k, v in result.items()}
    return result

# ------------------------------------------------------------
# 6) 학습 설정 (Training Arguments)
# ------------------------------------------------------------
args = Seq2SeqTrainingArguments(
    output_dir="./kobart-sum",          # 모델 저장 경로
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    num_train_epochs=5,
    logging_steps=10,

    eval_strategy="epoch",        # epoch마다 평가
    save_strategy="epoch",              # epoch마다 모델 저장

    # ✅ generate() 관련 옵션들 (여기!)
    predict_with_generate=True,
    generation_max_length=80,
    # 요약을 생성할 때 몇 개의 “경로(문장 후보)”를 동시에 탐색할지를 결정하는 옵션
    generation_num_beams=5,


    load_best_model_at_end=True,        # 가장 좋은 모델 자동 로드
    metric_for_best_model="rougeL",     # ROUGE-L 기준
    greater_is_better=True,             # 값이 클수록 좋음

    report_to="none",                   # wandb/logging 비활성
)

# ------------------------------------------------------------
# 7) Trainer 객체 생성
# ------------------------------------------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ------------------------------------------------------------
# 8) 학습 시작
# ------------------------------------------------------------
trainer.train()


C:\Users\ekfla\AppData\Local\Temp\ipykernel_27004\1280095923.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,5.106228,0.000000,0.000000,0.000000,0.000000
2,No log,5.801389,0.000000,0.000000,0.000000,0.000000
3,No log,5.723828,0.000000,0.000000,0.000000,0.000000
4,No log,5.379510,0.000000,0.000000,0.000000,0.000000
5,No log,5.094711,0.000000,0.000000,0.000000,0.000000


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warn

TrainOutput(global_step=5, training_loss=0.9023805618286133, metrics={'train_runtime': 21.8585, 'train_samples_per_second': 0.457, 'train_steps_per_second': 0.229, 'total_flos': 3048682291200.0, 'train_loss': 0.9023805618286133, 'epoch': 5.0})

In [45]:

# ------------------------------------------------------------
# 9) 검증 데이터 평가
# ------------------------------------------------------------
metrics = trainer.evaluate()
print("\n[Validation Metrics]", metrics)

# ------------------------------------------------------------
# 10) 파인튜닝된 모델로 직접 요약 생성 테스트
# ------------------------------------------------------------
test_text = "과학기술정보통신부는 초거대 AI 연구 인프라 지원을 강화한다고 밝혔다. 스타트업 대상으로 GPU 리소스를 확대 제공할 계획이다."
inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=max_input_len
)

inputs.pop("token_type_ids", None)  # 완전 제거
# 입력 인코딩 후 모델에 전달
gen_ids = model.generate(
    **inputs.to(model.device),
    max_length=80,
    num_beams=4,
    do_sample=False,
)

print("\n[생성 요약]")
print(tokenizer.decode(gen_ids[0], skip_special_tokens=True))



[Validation Metrics] {'eval_loss': 5.106228351593018, 'eval_rouge1': 0.0, 'eval_rouge2': 0.0, 'eval_rougeL': 0.0, 'eval_rougeLsum': 0.0, 'eval_runtime': 2.2816, 'eval_samples_per_second': 0.438, 'eval_steps_per_second': 0.438, 'epoch': 5.0}

[생성 요약]
정부는 초거대 AI 연구 인프라 지원을 강화한다고 밝혔다.
